# 06 — Long-Term Memory (Catalog Vector Search)

Tests `memory/lt_memory.py`:
- Embedding model loads correctly
- `precompute_embedding()` produces the right vector shape
- `search_catalog()` returns products with expected fields
- Pre-computed vector path works
- Allergen-relevant queries surface appropriate products

> **Requires**: Qdrant + env vars + catalog ingested.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

In [ ]:
from memory.lt_memory import precompute_embedding, search_catalog, encoder
from utils.config import QDRANT_EMBEDDING_DIM, LT_SEARCH_TOP_K

## 1. Encoder loaded

In [ ]:
assert encoder is not None, 'Encoder singleton must be initialized'
print('Encoder type:', type(encoder).__name__)
print('Encoder loaded: PASSED')

## 2. precompute_embedding shape

In [ ]:
vector = precompute_embedding('chocolate birthday cake')

print(f'Vector type : {type(vector)}')
print(f'Vector length: {len(vector)}')
print(f'First 5 values: {vector[:5]}')

assert isinstance(vector, list), 'Embedding should be a list'
assert len(vector) == QDRANT_EMBEDDING_DIM, \
    f'Expected {QDRANT_EMBEDDING_DIM} dims, got {len(vector)}'
assert all(isinstance(v, float) for v in vector[:10]), 'All values should be floats'
print('precompute_embedding: PASSED')

## 3. search_catalog — basic query

In [ ]:
results = search_catalog('birthday cake', top_k=5)

print(f'Results returned: {len(results)}')
for r in results:
    print(f"  [{r.get('category','?')}] {r.get('name','?')} — {r.get('price','?')}")

assert len(results) > 0, 'Search should return results'
assert len(results) <= 5, 'Should not exceed requested top_k'

# Each result must have expected payload fields
for r in results:
    assert 'name' in r, f'Missing name in result: {r}'
    assert 'price' in r, f'Missing price in result: {r}'
    assert 'category' in r, f'Missing category in result: {r}'
    assert 'product_url' in r, f'Missing product_url in result: {r}'

print('search_catalog basic: PASSED')

## 4. Pre-computed vector path

In [ ]:
query_vector = precompute_embedding('rose bouquet for anniversary')

results_via_query  = search_catalog('rose bouquet for anniversary', top_k=5)
results_via_vector = search_catalog('rose bouquet for anniversary', top_k=5, query_vector=query_vector)

names_q = [r['name'] for r in results_via_query]
names_v = [r['name'] for r in results_via_vector]

print('Via query  :', names_q)
print('Via vector :', names_v)

# Results should be identical (same vector either way)
assert names_q == names_v, \
    'Pre-computed vector path should yield the same results as on-the-fly embedding'
print('Pre-computed vector path: PASSED')

## 5. Semantic relevance check

In [ ]:
cake_results   = search_catalog('chocolate cake',    top_k=5)
flower_results = search_catalog('rose flower gift',  top_k=5)

cake_categories   = [r.get('category', '') for r in cake_results]
flower_categories = [r.get('category', '') for r in flower_results]

print('Cake query categories  :', cake_categories)
print('Flower query categories:', flower_categories)

# Cake query should predominantly return cakes
assert cake_categories.count('cakes') >= 2, \
    f'Expected mostly cakes for cake query, got: {cake_categories}'

# Flower query should predominantly return flowers
assert flower_categories.count('flowers') >= 2, \
    f'Expected mostly flowers for flower query, got: {flower_categories}'

print('Semantic relevance: PASSED')

## 6. Default top_k from config

In [ ]:
results = search_catalog('gift for mother')
print(f'Default top_k results: {len(results)} (config says {LT_SEARCH_TOP_K})')
assert len(results) <= LT_SEARCH_TOP_K
print('Default top_k: PASSED')